# Data Exploration

In [ ]:
import pandas as pd
import glob

files = glob.glob("data/raw/*.csv")
print(files)

# Load one file first to see the actual structure
sample = pd.read_csv("data/raw/raw_test_bernie.csv")
print(sample.shape)
print(sample.columns.tolist())
print(sample.head())

In [ ]:
splits = ['train', 'val', 'test']
targets = ['trump', 'biden', 'bernie']

count = 0

for split in splits:
    for target in targets:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}: {df.shape[0]} rows")
        print(df['Stance'].value_counts())
        print()

        count += df.shape[0]

print(f"count {count}")

In [ ]:
train_dfs = []
for target in targets:
    df = pd.read_csv(f"data/raw/raw_train_{target}.csv")
    train_dfs.append(df)

train_combined = pd.concat(train_dfs, ignore_index=True)
train_combined.to_csv("data/processed/train_combined.csv", index=False)

In [ ]:
for split in ['train', 'val', 'test']:
    for target in ['trump', 'biden', 'bernie']:
        df = pd.read_csv(f"data/raw/raw_{split}_{target}.csv")
        print(f"{split}_{target}:")
        print(df['Stance'].value_counts(normalize=True))
        print()

In [ ]:
train = pd.read_csv("data/processed/train_combined.csv")
test_dfs = [pd.read_csv(f"data/raw/raw_test_{t}.csv") for t in ['trump','biden','bernie']]
test = pd.concat(test_dfs, ignore_index=True)

overlap = set(train['Tweet']) & set(test['Tweet'])
print(len(overlap))

In [ ]:
from transformers import AutoTokenizer
from src.data import format_prompt

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

lengths = train.apply(lambda row: len(tokenizer.encode(format_prompt(row['Tweet'], row['Target']))), axis=1)
print(lengths.describe())

# Environment Setup

In [1]:
# Setup cell - rerun this after any kernel disconnect/reconnect

import os

REPO_DIR = "/content/political-stance-detection"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/meghna-adduri/political-stance-detection.git {REPO_DIR}
    %cd {REPO_DIR}

!pip install transformers accelerate bitsandbytes wandb tqdm -q

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import wandb
import logging

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

import sys
sys.path.append(REPO_DIR)
from src.data import load_combined, format_prompt
from src.baseline import load_model, get_prediction, parse_label, run_baseline, evaluate_and_log

# Reload model
model, tokenizer = load_model()

# Reload prior results, if they exist
try:
    results = pd.read_csv("data/processed/zeroshot_predictions.csv")
    print(f"Loaded existing results: {len(results)} rows")
except FileNotFoundError:
    print("No existing results file found, run run_baseline() to generate one")

# Confirm GPU is actually attached
!nvidia-smi

/content/political-stance-detection
Already up to date.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded existing results: 2157 rows
Sat Aug 15 03:03:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             33W /   70W |    8745MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

In [2]:
!pwd
!ls

/content/political-stance-detection
data	   pyproject.toml  requirements.txt  src
notebooks  README.md	   scratch.ipynb     tests


In [3]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: meghna-adduri (meghna-adduri-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!git config --global user.email "meghna.adduri07@gmail.com"
!git config --global user.name "meghna-adduri"

In [5]:
!git pull

Already up to date.


# Zero-shot Baseline

In [ ]:
from getpass import getpass

results = run_baseline(model, tokenizer)

!git add data/processed/zeroshot_predictions.csv
!git commit -m "Add zero-shot baseline predictions (fixed truncation)"

token = getpass("Enter your GitHub token: ")
!git push https://{token}@github.com/meghna-adduri/political-stance-detection.git main

acc, macro_f1 = evaluate_and_log(results, run_name="zeroshot-qwen2.5-7b")
print(f"Zero-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

In [ ]:
print(f"Zero-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

In [ ]:
print(results['prediction'].value_counts())

In [ ]:
results[['Tweet', 'Target', 'Stance', 'prediction']].sample(15)

In [ ]:
!git add data/processed/zeroshot_predictions.csv
!git commit -m "Add zero-shot baseline predictions"

# Debugging UNKNOWN predictions

In [ ]:
sample_unknown = results[results['prediction'] == 'UNKNOWN'].sample(15, random_state=42)
sample_favor = results[results['prediction'] == 'FAVOR'].sample(15, random_state=42)
sample_against = results[results['prediction'] == 'AGAINST'].sample(15, random_state=42)

debug_rows = pd.concat([sample_unknown, sample_favor, sample_against])

debug_results = []
for _, row in debug_rows.iterrows():
    raw, pred = get_prediction(model, tokenizer, row['Tweet'], row['Target'])
    debug_results.append({
        'tweet': row['Tweet'][:80],
        'target': row['Target'],
        'original_prediction': row['prediction'],
        'raw_response': raw,
        'response_length_tokens': len(tokenizer.encode(raw))
    })

debug_df = pd.DataFrame(debug_results)

In [ ]:
print(debug_df.groupby('original_prediction')['response_length_tokens'].describe())

In [ ]:
pd.set_option('display.max_colwidth', None)
print(debug_df[['original_prediction', 'raw_response']].to_string())

In [ ]:
print(debug_df['raw_response'].str.strip().str.upper().isin(['FAVOR', 'AGAINST']).mean())

# Finding Examples for Few-Shot

In [6]:
train = pd.read_csv("data/processed/train_combined.csv")

targets = ['Donald Trump', 'Joe Biden', 'Bernie Sanders']
stances = ['FAVOR', 'AGAINST']

candidates = {}
for target in targets:
    for stance in stances:
        key = f"{target}_{stance}"
        candidates[key] = train[
            (train['Target'] == target) &
            (train['Stance'] == stance)
        ]
        print(f"{key}: {len(candidates[key])} candidates")

Donald Trump_FAVOR: 2937 candidates
Donald Trump_AGAINST: 3425 candidates
Joe Biden_FAVOR: 2552 candidates
Joe Biden_AGAINST: 3254 candidates
Bernie Sanders_FAVOR: 2858 candidates
Bernie Sanders_AGAINST: 2198 candidates


In [7]:
shortlists = {}
for key, df in candidates.items():
    df = df.copy()
    df['word_count'] = df['Tweet'].str.split().str.len()
    filtered = df[(df['word_count'] >= 8) & (df['word_count'] <= 25)]
    shortlists[key] = filtered.sample(min(5, len(filtered)), random_state=42)

for key, shortlist in shortlists.items():
    print(f"\n=== {key} ===")
    for _, row in shortlist.iterrows():
        print(row['Tweet'])
        print('---')


=== Donald Trump_FAVOR ===
Trump sends 5,200 troops to Mexico border as caravan advances - Reuters THEYRE NOT COMING IN #POTUS45 IS MAN OF HIS WORD! #VoteDemsOut #PatriotsEffectingChange #PatriotsVoteRedNov6
---
#Democrat #Democrats THIS is the crux of Trump placing them in beds and checking DNA versus Obama's Cages. Where are The Parents?
---
But yes, I believe in both too! And I believe in you! #Trump
---
Dear Mr. President, thank you for signing this into law #KeepAmericaGreat #President #Nowletsgetyoureelected #republican #Trump
---
Im convinced this works in Trumps favor. Master of spectacle. Gets lulz from his tribe and engraves ours. Also, if goes viral over social media.
---

=== Donald Trump_AGAINST ===
Those Trumpers know something is wrong with them so they are sensitive. #Fakebonespurs #AlertTheDayCareStaff #GetPenceFirst #TrumpChristmasShutdown
---
@realDonaldTrump @WhiteHouse Winning much, you stupid Fu4ks?? Or too busy try to bury #trumpepsteinrapists???
---
Sources con

In [8]:
selected = {
    'Donald Trump_FAVOR': shortlists['Donald Trump_FAVOR'].iloc[3],
    'Donald Trump_AGAINST': shortlists['Donald Trump_AGAINST'].iloc[2],
    'Joe Biden_FAVOR': shortlists['Joe Biden_FAVOR'].iloc[3],
    'Joe Biden_AGAINST': shortlists['Joe Biden_AGAINST'].iloc[4],
    'Bernie Sanders_FAVOR': shortlists['Bernie Sanders_FAVOR'].iloc[0],
    'Bernie Sanders_AGAINST': shortlists['Bernie Sanders_AGAINST'].iloc[2],
}

In [18]:
import json

few_shot_examples = []
for key, row in selected.items():
    target, stance = key.rsplit('_', 1)
    few_shot_examples.append({
        "tweet": row['Tweet'],
        "target": row['Target'],
        "label": row['Stance']
    })

with open("data/processed/few_shot_examples.json", "w") as f:
    json.dump(few_shot_examples, f, indent=2)

print(f"Saved {len(few_shot_examples)} examples")

Saved 6 examples


In [10]:
import json

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

with open("data/processed/few_shot_examples.json") as f:
    few_shot_examples = json.load(f)

from src.data import format_few_shot_prompt

lengths = train.apply(
    lambda row: len(tokenizer.encode(
        format_few_shot_prompt(row['Tweet'], row['Target'], few_shot_examples)
    )),
    axis=1
)
print(lengths.describe())

count    17224.000000
mean       330.619194
std         16.784980
min        296.000000
25%        317.000000
50%        330.000000
75%        344.000000
max        421.000000
dtype: float64


# Few-Shot Run

In [6]:
from src.data import format_few_shot_prompt
import json

with open("data/processed/few_shot_examples.json") as f:
    few_shot_examples = json.load(f)

test_df = load_combined('test')

diagnostic_sample = (
    test_df.groupby('Target', group_keys=False)
    .apply(lambda x: x.sample(5, random_state=42))
)

# Run few-shot prediction on just this small sample
debug_results = []
for _, row in diagnostic_sample.iterrows():
    prompt = format_few_shot_prompt(row['Tweet'], row['Target'], few_shot_examples)
    raw, pred = get_prediction(model, tokenizer, prompt)
    debug_results.append({
        'tweet': row['Tweet'][:80],
        'target': row['Target'],
        'true_label': row['Stance'],
        'prediction': pred,
        'raw_response': raw,
        'response_length_tokens': len(tokenizer.encode(raw)),
        'correct': pred == row['Stance']
    })

debug_df = pd.DataFrame(debug_results)

/tmp/ipykernel_2335/247144092.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(5, random_state=42))


In [7]:
print(debug_df['prediction'].value_counts())
print(debug_df.groupby('prediction')['response_length_tokens'].describe())

prediction
AGAINST    10
FAVOR       5
Name: count, dtype: int64
            count  mean  std  min  25%  50%  75%  max
prediction                                           
AGAINST      10.0   2.0  0.0  2.0  2.0  2.0  2.0  2.0
FAVOR         5.0   3.0  0.0  3.0  3.0  3.0  3.0  3.0


In [8]:
print(debug_df['correct'].mean())

0.9333333333333333


In [9]:
pd.set_option('display.max_colwidth', None)
print(debug_df[['tweet', 'target', 'true_label', 'prediction', 'raw_response']].to_string())

                                                                               tweet          target true_label prediction raw_response
0   I'm gonnarigthe primary against Bernie by... being a POC, & voting in it! being   Bernie Sanders    AGAINST      FAVOR        FAVOR
1   The sad part is that Bernie went ahead and ordered the soup anyway. #EndTheDuopo  Bernie Sanders    AGAINST    AGAINST      AGAINST
2   Get real, 'ol Bernie is 78 with one foot on a banana peel, you better be concern  Bernie Sanders    AGAINST    AGAINST      AGAINST
3                              This right here is why I could NEVER VOTE FOR SANDERS  Bernie Sanders    AGAINST    AGAINST      AGAINST
4    #BernieSanders and #Cult45 use the same tactics you can barely tell them apart.  Bernie Sanders    AGAINST    AGAINST      AGAINST
5            If a Trump were any dumber, hed be a brick. What a pathetic dolt! #Dems    Donald Trump    AGAINST    AGAINST      AGAINST
6   In the past 6 weeks 26 million Americans los

In [10]:
print(format_few_shot_prompt(diagnostic_sample.iloc[0]['Tweet'], diagnostic_sample.iloc[0]['Target'], few_shot_examples))

Tweet: Dear Mr. President, thank you for signing this into law #KeepAmericaGreat #President #Nowletsgetyoureelected #republican #Trump
Target: Donald Trump
Answer: FAVOR

Tweet: Sources confirm that Trump is manipulating the stock markets. Which means the rich get richer. NOT YOU. #Winning
Target: Donald Trump
Answer: AGAINST

Tweet: This and so many other civil rights causes Biden has championed thru the decades is why he's winning in the South. We remember.
Target: Joe Biden
Answer: FAVOR

Tweet: lol. #Biden's defense is he was too incompetent as VP to stop the escalation of the war in #Afghanistan #DemDebate
Target: Joe Biden
Answer: AGAINST

Tweet: #Bernie has some really good plans so you lost me at the starting gate.
Target: Bernie Sanders
Answer: FAVOR

Tweet: Crush. Kill. Destroy. Hell No to socialist takeover of the Democratic Party. #MichiganPrimary #BernieSanders
Target: Bernie Sanders
Answer: AGAINST

Tweet: I'm gonnarigthe primary against Bernie by... being a POC, & voting

In [6]:
# Real run on full dataset

from src.few_shot import run_few_shot, load_few_shot_examples
from src.baseline import evaluate_and_log

few_shot_examples = load_few_shot_examples()

# Push results to GitHub immediately after generation, before eval,
# same safety pattern as zero-shot tonight
results = run_few_shot(model, tokenizer, few_shot_examples)

!git add data/processed/fewshot_predictions.csv
!git commit -m "Add few-shot baseline predictions"

from getpass import getpass
token = getpass("Enter your GitHub token: ")
!git push https://{token}@github.com/meghna-adduri/political-stance-detection.git main

# compute and log metrics
acc, macro_f1 = evaluate_and_log(results, run_name="fewshot-6ex-qwen2.5-7b")
print(f"Few-shot accuracy: {acc:.3f}, macro-F1: {macro_f1:.3f}")

100%|██████████| 2157/2157 [32:10<00:00,  1.12it/s]


[main a4ef580] Add few-shot baseline predictions
 1 file changed, 2158 insertions(+)
 create mode 100644 data/processed/fewshot_predictions.csv
Enter your GitHub token: ··········
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 170.71 KiB | 4.88 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/meghna-adduri/political-stance-detection.git
   09aa437..a4ef580  main -> main


accuracy,▁
accuracy_Bernie Sanders,▁
accuracy_Donald Trump,▁
accuracy_Joe Biden,▁
macro_f1,▁
macro_f1_Bernie Sanders,▁
macro_f1_Donald Trump,▁
macro_f1_Joe Biden,▁
total_examples,▁
unknown_predictions,▁
accuracy,0.76252


Few-shot accuracy: 0.763, macro-F1: 0.752


# Useful git commands

In [8]:
from getpass import getpass
token = getpass("Enter your GitHub token: ")
!git push https://{token}@github.com/meghna-adduri/political-stance-detection.git main

Enter your GitHub token: ··········
Everything up-to-date
